In [25]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
import time
from src.backtest.database.mongodb_klg import MongoDBKLG
load_dotenv()
local_connection = MongoClient("mongodb://localhost:27017")
klg_connection = MongoClient(os.environ.get("ENTITIES_DB"))
protocol_connection = MongoClient(os.environ.get("DAPP_INFO_DB"))
klg_database = klg_connection["knowledge_graph"]
local_database = local_connection["blockchain_etl"]
protocol_database = protocol_connection["SmartContractLabel"]
atlas_klg = MongoDBKLG("mongodb+srv://@cluster0.vp4q6pi.mongodb.net/")
local_klg = MongoDBKLG("mongodb://localhost:27017")
klg = MongoDBKLG(os.environ.get("ENTITIES_DB"))

# Export balance history
- Run file export_balance_wallets.py
- Dowwnload token balance and token transfer in https://chainbase.com/ and store in balance folder

In [26]:
files = [f for f in os.listdir('src/backtest/balance') if os.path.isfile(f'src/backtest/balance/{f}')]
block_per_second = {
    "0x38": 3,
    "0x1": 12,
    "0x89": 2.1,
    "0xa4b1": 0.3
}
timestamps = {
    "0x1":{
        "1727654411": 20859741
    },
    "0xa4b1":{
        "1726830680": 255464462
    },
    "0x89":{
        "1722830071": 60208605
    },
    "0x38":{
        "1721189933": 40541960
    }
}
chain_collection = {
    "0x38": "bnb",
    "0x89": "poly",
    "0x1": "eth",
    "0xa4b1": "arb"
}

def round_timestamp(timestamp, round_time=86400):
    timestamp = int(timestamp)
    timestamp_unit_day = timestamp / round_time
    recover_to_unit_second = int(timestamp_unit_day) * round_time
    return recover_to_unit_second

def calculate_block_by_timestamp(timestamp_, chain_id):
    second_per_block = block_per_second.get(chain_id)
    stable_time, block = 0, 0
    for key, value_ in timestamps.get(chain_id).items():
        stable_time = int(key)
        block = value_

    block_change = (stable_time - timestamp_)/second_per_block
    return int(block + block_change)

def calculate_timestamp_by_block_number(block_number_, chain_id):
    second_per_block = block_per_second.get(chain_id)
    stable_time, block = 0, 0
    for key, value_ in timestamps.get(chain_id).items():
        stable_time = int(key)
        block = value_

    time_change = (block - block_number_)*second_per_block
    return int(stable_time + time_change)

def get_price(key_token, token_data, block_timestamp):
    if key_token not in token_data:
        return 0
    price = None
    tmp = time.time()
    for timestamp in token_data[key_token].get("priceChangeLogs", {}):
        if int(timestamp) < tmp and int(timestamp) >= block_timestamp:
            price = token_data[key_token].get("priceChangeLogs", {}).get(timestamp)
            tmp = int(timestamp)

    if not price:
        price = token_data[key_token].get("price",0) or 0
    return price

In [27]:
import pandas as pd
chain = {
    "arbitrum": "0xa4b1",
    "bsc":"0x38",
    "polygon": "0x89",
    "ethereum":"0x1",
}
result = {}
token_data = {}
for f in files:
    print(f"file: {f}")
    df = pd.read_csv(f'src/backtest/balance/{f}')
    chain_id = None
    for i in chain:
        if i in f:            
            chain_id = chain[i]
    if not chain_id:
        continue
    print(chain_id)
    tokens = [f"{chain_id}_{token}" for token in df["contract_address"] if f"{chain_id}_{token}" not in token_data]
    tokens = list(set(tokens))
    print(f"tokens: {len(tokens)}")
    cursor = klg_database["smart_contracts"].find({"_id":{"$in": tokens}})
    for item in cursor:
        key_token = item.get("_id")
        token_data[key_token] = {
            "decimals": item.get("decimals"),
            "price": item.get("price", 0),
            "priceChangeLogs": item.get("priceChangeLogs",{})
        }
    for index, row in df.iterrows():
        token = row["contract_address"]
        amount = float(row["amount"])
        wallet = row["wallet_address"]
        block_number = row["block_number"]
        block_timestamp = calculate_timestamp_by_block_number(block_number, chain_id)
        if wallet not in result:
            result[wallet] = {
                "_id": wallet,
                "address": wallet,
                "tokenChangeLogs": {}
            }
        key_token = f"{chain_id}_{token}"
        if key_token not in result[wallet]["tokenChangeLogs"]:
            result[wallet]["tokenChangeLogs"][key_token]={}
        price = get_price(key_token, token_data, block_timestamp)
        decimals = token_data.get(key_token, {}).get("decimals", 18) or 18
        amount_in_usd = float(amount) * price/10**decimals
        result[wallet]["tokenChangeLogs"][key_token][str(block_timestamp)] = {
            "amount": amount/10**decimals,
            "valueInUSD": amount_in_usd
        }
update_data = list(result.values())
local_klg.update_multichain_wallets(update_data)

file: polygon_balance.csv
0x89
tokens: 16
file: ethereum_balance_5.csv
0x1
tokens: 13
file: ethereum_balance_1.csv
0x1
tokens: 1
file: arbitrum_balance.csv
0xa4b1
tokens: 25
file: ethereum_balance_9.csv
0x1
tokens: 2
file: ethereum_balance_7.csv
0x1
tokens: 1
file: bsc_balance.csv
0x38
tokens: 28
file: ethereum_balance_8.csv
0x1
tokens: 1
file: ethereum_balance_11.csv
0x1
tokens: 0
file: ethereum_balance_10.csv
0x1
tokens: 1
file: ethereum_balance_2.csv
0x1
tokens: 0
file: ethereum_balance_3.csv
0x1
tokens: 0
file: ethereum_balance_6.csv
0x1
tokens: 0
file: ethereum_balance_4.csv
0x1
tokens: 0


In [29]:
wallets = [item["_id"] for item in local_connection["knowledge_graph"]["multichain_wallets"].find()]

# Daily transaction interactions

In [15]:
update_data = {}
result = {}
for chain_id, prefix in chain_collection.items():
    tx_collection = local_database[f"{prefix}_transactions"]
    tf_collection = local_database[f"{prefix}_tf_events"]
    for tx in tx_collection.find({"from_address":{"$in": wallets}}):
        address = tx.get("from_address")
        if address not in result:
            result[address] = {
                "_id": address,
                "address": address,
                "dailyNumberOfTransactionsInEachChain":{}
            }
        if chain_id not in result[address]["dailyNumberOfTransactionsInEachChain"]:
            result[address]["dailyNumberOfTransactionsInEachChain"][chain_id] = {}
        r_timestamp = round_timestamp(tx.get('block_timestamp'))
        if str(r_timestamp) not in result[address]["dailyNumberOfTransactionsInEachChain"][chain_id]:
            result[address]["dailyNumberOfTransactionsInEachChain"][chain_id][str(r_timestamp)] = []
        result[address]["dailyNumberOfTransactionsInEachChain"][chain_id][str(r_timestamp)].append(tx.get("hash"))

    for event in tf_collection.find({"from_address":{"$in": wallets}}):
        address = event.get("from_address")
        if address not in result:
            result[address] = {
                "_id": address,
                "address": address,
                "dailyNumberOfTransactionsInEachChain":{}
            }
        if chain_id not in result[address]["dailyNumberOfTransactionsInEachChain"]:
            result[address]["dailyNumberOfTransactionsInEachChain"][chain_id] = {}
            
        r_timestamp = round_timestamp(calculate_timestamp_by_block_number(event.get('block_number'), chain_id))
        if str(r_timestamp) not in result[address]["dailyNumberOfTransactionsInEachChain"][chain_id]:
            result[address]["dailyNumberOfTransactionsInEachChain"][chain_id][str(r_timestamp)] = []

        if event.get("transaction_hash") not in result[address]["dailyNumberOfTransactionsInEachChain"][chain_id][str(r_timestamp)]:
            result[address]["dailyNumberOfTransactionsInEachChain"][chain_id][str(r_timestamp)].append(tx.get("transaction_hash"))
            
update_data = {}
for address, value in result.items():
    if address not in update_data:
        update_data[address] = {}

    #dailyNumberOfTransactions
    update_data[address]["dailyNumberOfTransactions"] = {}
    for chain_id, logs in value["dailyNumberOfTransactionsInEachChain"].items():
        for timestamp, tx_list in logs.items():
            if timestamp not in update_data[address]["dailyNumberOfTransactions"]:
                update_data[address]["dailyNumberOfTransactions"][timestamp] = 0
            update_data[address]["dailyNumberOfTransactions"][timestamp] += len(tx_list)

In [16]:
update_list = []
for i in update_data:
    tmp = {
        "_id": i,
        "address": i,
        "dailyNumberOfTransactions": update_data[i]["dailyNumberOfTransactions"]
    }
    update_list.append(tmp)
local_klg.update_multichain_wallets(update_list)

# Daily transaction amount

In [30]:
tokens = []
for chain_id, prefix in chain_collection.items():
    tx_collection = local_database[f"{prefix}_transactions"]
    tf_collection = local_database[f"{prefix}_tf_events"]
    tokens += [f"{chain_id}_{event.get('contract_address')}" for event in tf_collection.find({"to_address":{"$in": wallets}})]
    tokens += [f"{chain_id}_0x0000000000000000000000000000000000000000"]
tokens = list(set(tokens))
tokens = [token for token in tokens if token not in token_data]
for token in klg_database["smart_contracts"].find({"_id": {"$in": tokens}}):
    key_token = item.get("_id")
    token_data[key_token] = {
                "decimals": item.get("decimals"),
                "price": item.get("price", 0),
                "priceChangeLogs": item.get("priceChangeLogs",{})}

In [32]:
result = {}
for chain_id, prefix in chain_collection.items():
    begin = time.time()
    print("transaction")
    tx_collection = local_database[f"{prefix}_transactions"]
    tf_collection = local_database[f"{prefix}_tf_events"]
    for tx in tx_collection.find({"to_address":{"$in": wallets}}):
        address = tx.get("to_address")
        if address not in result:
            result[address] = {
                "_id": address,
                "address": address,
                "dailyTransactionAmountsInEachChain":{}
            }
        if chain_id not in result[address]["dailyTransactionAmountsInEachChain"]:
            result[address]["dailyTransactionAmountsInEachChain"][chain_id] = {}
        r_timestamp = round_timestamp(tx.get('block_timestamp'))
        if str(r_timestamp) not in result[address]["dailyTransactionAmountsInEachChain"][chain_id]:
            result[address]["dailyTransactionAmountsInEachChain"][chain_id][str(r_timestamp)] = 0

        token_key = f'{chain_id}_0x0000000000000000000000000000000000000000'
        if token_key not in token_data:
            cursor = klg_database["smart_contracts"].find_one({"_id": token_key})
            key_token = item.get("_id")
            token_data[key_token] = {
                "decimals": item.get("decimals"),
                "price": item.get("price", 0),
                "priceChangeLogs": item.get("priceChangeLogs",{})}
        price = get_price(token_key, token_data, tx.get("block_timestamp"))
        result[address]["dailyTransactionAmountsInEachChain"][chain_id][str(r_timestamp)] += float(tx.get("value", 0))*price/10**18
        
    print(f"event: {time.time() - begin}")
    for event in tf_collection.find({"to_address":{"$in": wallets}}):
        address = event.get("to_address")
        if address not in result:
            result[address] = {
                "_id": address,
                "address": address,
                "dailyTransactionAmountsInEachChain":{}
            }
        if chain_id not in result[address]["dailyTransactionAmountsInEachChain"]:
            result[address]["dailyTransactionAmountsInEachChain"][chain_id] = {}
        
        token_key = f'{chain_id}_{event.get("contract_address")}'
        timestamp = calculate_timestamp_by_block_number(event.get('block_number'), chain_id)
        r_timestamp = round_timestamp(timestamp)
        if str(r_timestamp) not in result[address]["dailyTransactionAmountsInEachChain"][chain_id]:
            result[address]["dailyTransactionAmountsInEachChain"][chain_id][str(r_timestamp)] = 0

        price = get_price(token_key, token_data, timestamp)
        result[address]["dailyTransactionAmountsInEachChain"][chain_id][str(r_timestamp)] += event.get("value") * price
    print(f"all: {time.time() - begin}")

transaction
event: 72.25524377822876
all: 72.37184739112854
transaction
event: 40.256375312805176
all: 40.472630977630615
transaction
event: 118.54675054550171
all: 118.75785398483276
transaction
event: 93.75236201286316
all: 94.19671678543091


In [20]:
import json
# with open("daily_tx_amount.json", "r") as f:
#     data = json.loads(f.read())
data = result
update_data = []
for i in data:
    tmp = {
        "_id": i,
        "address": i,
        "dailyTransactionAmounts":{}
    }
    for chain, logs in data[i]["dailyTransactionAmountsInEachChain"].items():
        for timestamp, value in logs.items():
            if not value:
                continue
            if timestamp not in tmp["dailyTransactionAmounts"]:
                tmp["dailyTransactionAmounts"][timestamp] = 0
            tmp["dailyTransactionAmounts"][timestamp] += value
    update_data.append(tmp)
# local_klg.update_multichain_wallets(update_data)

# Project interaction

In [9]:
contract = []
for chain_id, prefix in chain_collection.items():
    tx_collection = local_database[f"{prefix}_transactions"]
    tf_collection = local_database[f"{prefix}_tf_events"]
    contract += [f"{chain_id}_{event.get('contract_address')}" for event in tf_collection.find({"$or":[{"to_address":{"$in": wallets}}, {"from_address":{"$in": wallets}}]})]
    contract += [f"{chain_id}_{event.get('to_address')}" for event in tx_collection.find({"from_address":{"$in": wallets}})]
contract  = list(set(contract))

In [10]:
contract_data = {}
projects = []
for c in klg_database["smart_contracts"].find({"_id": {"$in": contract}}):
    project = None
    if c.get("projectDapp"):
        project = c.get("projectDapp")

    elif c.get("project"):
        project = c.get("project")

    if not project:
        continue
    projects.append(project)
    contract_data[c.get("_id")] = {
        "project": project,
        "reputation": 0,
        "category": c.get("category")
    }

projects  = list(set(projects))
print("number of project: ", len(projects))
re_projects = []
for p in klg_database["projects"].find({"_id": {"$in": projects}}):
    if p.get("tvl", 0) >= 10000000:
        re_projects.append(p.get("_id"))

for k, v in contract_data.items():
    if v["project"] in re_projects:
        v["reputation"] = 1

number of project:  920


In [11]:
project_dict = {}
for item in protocol_database["smart_contracts"].find({"_id": {"$in":contract}}):
    if contract_data[item.get("_id")].get("category"):
        continue
    else:
       contract_data[item.get("_id")]['category'] =  item.get("category")

In [12]:
result = {}
for chain_id, prefix in chain_collection.items():
    tx_collection = local_database[f"{prefix}_transactions"]
    tf_collection = local_database[f"{prefix}_tf_events"]

    for event in tf_collection.find({"$or":[{"to_address":{"$in": wallets}}, {"from_address":{"$in": wallets}}]}):
        contract_key = f"{chain_id}_{event.get('contract_address')}"
        if contract_key not in contract_data:
            continue
        reputable = contract_data.get(contract_key).get("reputation")
        pj = contract_data.get(contract_key).get("project")
        category = contract_data.get(contract_key).get("category") or "NaN"
        address = event.get("from_address")
        if address not in wallets:
            address  = event.get("to_address")

        if address not in result:
            result[address] = {
                    'frequencyOfDappTransactionsInEachChain': {chain_id: {}},
                    'numberOfInteractedDappsInEachChain': {chain_id: {}},
                    'typesOfInteractedDappsInEachChain': {chain_id: {}},
                    'numberOfReputableDappsInEachChain': {chain_id: {}},
                }
        else:
            for k, v in result[address].items():
                if chain_id not in v:
                    v[chain_id] = {}
                    
        block_timestamp = calculate_timestamp_by_block_number(event.get('block_number'), chain_id)
        r_timestamp = round_timestamp(block_timestamp)
        key = str(r_timestamp)
        for k, v in result[address].items():
            for cid, logs in v.items():
                if key not in logs:
                    logs[key] = []
        
        #frequencyOfDappTransactionsInEachChain
        if event.get("transaction_hash") not in result[address]['frequencyOfDappTransactionsInEachChain'][chain_id][key]:
            result[address]['frequencyOfDappTransactionsInEachChain'][chain_id][key].append(event.get("transaction_hash"))
        #numberOfReputableDappsInEachChain
        if reputable and pj not in result[address]['numberOfReputableDappsInEachChain'][chain_id][key]:
            result[address]['numberOfReputableDappsInEachChain'][chain_id][key].append(pj)
        #numberOfInteractedDappsInEachChain
        if pj not in result[address]['numberOfInteractedDappsInEachChain'][chain_id][key]:
            result[address]['numberOfInteractedDappsInEachChain'][chain_id][key].append(pj)
        #typesOfInteractedDappsInEachChain
        if category not in result[address]['typesOfInteractedDappsInEachChain'][chain_id][key]:
            result[address]['typesOfInteractedDappsInEachChain'][chain_id][key].append(category)


    for event in tx_collection.find({"from_address":{"$in": wallets}}):
        contract_key = f"{chain_id}_{event.get('to_address')}"
        if contract_key not in contract_data:
            continue
        reputable = contract_data.get(contract_key).get("reputation")
        pj = contract_data.get(contract_key).get("project")
        category = contract_data.get(contract_key).get("category") or "NaN"
        address = event.get("from_address")
        if address not in result:
            result[address] = {
                    'frequencyOfDappTransactionsInEachChain': {chain_id: {}},
                    'numberOfInteractedDappsInEachChain': {chain_id: {}},
                    'typesOfInteractedDappsInEachChain': {chain_id: {}},
                    'numberOfReputableDappsInEachChain': {chain_id: {}},
                }
        else:
            for k, v in result[address].items():
                if chain_id not in v:
                    v[chain_id] = {}
        
        r_timestamp = round_timestamp(event.get("block_timestamp"))
        key = str(r_timestamp)
        for k, v in result[address].items():
            for cid, logs in v.items():
                if key not in logs:
                    logs[key] = []
        #frequencyOfDappTransactionsInEachChain
        if event.get("transaction_hash") not in result[address]['frequencyOfDappTransactionsInEachChain'][chain_id][key]:
            result[address]['frequencyOfDappTransactionsInEachChain'][chain_id][key].append(event.get("transaction_hash"))
        #numberOfReputableDappsInEachChain
        if reputable and pj not in result[address]['numberOfReputableDappsInEachChain'][chain_id][key]:
            result[address]['numberOfReputableDappsInEachChain'][chain_id][key].append(pj)
        #numberOfInteractedDappsInEachChain
        if pj not in result[address]['numberOfInteractedDappsInEachChain'][chain_id][key]:
            result[address]['numberOfInteractedDappsInEachChain'][chain_id][key].append(pj)
        #typesOfInteractedDappsInEachChain
        if category not in result[address]['typesOfInteractedDappsInEachChain'][chain_id][key]:
            result[address]['typesOfInteractedDappsInEachChain'][chain_id][key].append(category)
update_data = {}
for address, value in result.items():
    if address not in update_data:
        update_data[address] = {}

    #frequencyOfDappTransactions
    update_data[address]["frequencyOfDappTransactions"] = {}
    for chain_id, logs in value["frequencyOfDappTransactionsInEachChain"].items():
        for timestamp, tx_list in logs.items():
            if timestamp not in update_data[address]["frequencyOfDappTransactions"]:
                update_data[address]["frequencyOfDappTransactions"][timestamp] = 0
            update_data[address]["frequencyOfDappTransactions"][timestamp] += len(tx_list)
    #numberOfReputableDapps
    update_data[address]["numberOfReputableDapps"] = {}
    for chain_id, logs in value["numberOfReputableDappsInEachChain"].items():
        for timestamp, pj_list in logs.items():
            if not pj_list:
                continue
            if timestamp not in update_data[address]["numberOfReputableDapps"]:
                update_data[address]["numberOfReputableDapps"][timestamp] = []
            for pj in pj_list:
                if pj not in update_data[address]["numberOfReputableDapps"][timestamp]:
                    update_data[address]["numberOfReputableDapps"][timestamp].append(pj)
    #numberOfInteractedDapps
    update_data[address]["numberOfInteractedDapps"] = {}
    for chain_id, logs in value["numberOfInteractedDappsInEachChain"].items():
        for timestamp, pj_list in logs.items():
            if not pj_list:
                continue
            if timestamp not in update_data[address]["numberOfInteractedDapps"]:
                update_data[address]["numberOfInteractedDapps"][timestamp] = []
            for pj in pj_list:
                if pj not in update_data[address]["numberOfInteractedDapps"][timestamp]:
                    update_data[address]["numberOfInteractedDapps"][timestamp].append(pj)
    #typesOfInteractedDapps
    update_data[address]["typesOfInteractedDapps"] = {}
    for chain_id, logs in value["typesOfInteractedDappsInEachChain"].items():
        for timestamp, pj_list in logs.items():
            if not pj_list:
                continue
            if timestamp not in update_data[address]["typesOfInteractedDapps"]:
                update_data[address]["typesOfInteractedDapps"][timestamp] = []
            for pj in pj_list:
                if pj not in update_data[address]["typesOfInteractedDapps"][timestamp]:
                    update_data[address]["typesOfInteractedDapps"][timestamp].append(pj)    

In [13]:
update_list = []
for i, value in update_data.items():
    tmp = {
        "_id": i,
        "address": i
    }
    tmp.update(value)
    update_list.append(tmp)
local_klg.update_multichain_wallets(update_list)

In [31]:
update_data = []
wallets = [wallet.get("address") for wallet in local_connection["knowledge_graph"]["multichain_wallets"].find()]
for wallet in local_connection["knowledge_graph"]["multichain_wallets_2"].find({"_id":{"$in": wallets}}):
    update_data.append({
        "_id": wallet.get("_id"),
        "dailyTransactionAmounts": wallet.get("dailyTransactionAmounts", {})
    })
local_klg.update_multichain_wallets(update_data)

# Liquidation
Run file export_liquidate_wallet_history.py

In [14]:
eth_db = {
    "0x38": "blockchain_etl",
    "0x1": "ethereum_blockchain_etl",
    "0x89": "polygon_blockchain_etl",
    "0xa4b1": "arbitrum_blockchain_etl"
}
result = {}
for chain_id, value in eth_db.items():
    database = local_connection[value]
    debtors = database["debtors"]
    for item in debtors.find({"_id":{"$in": wallets}}):
        address = item.get("_id")
        if address not in result:
            result[address] = {
                "liquidatedWallet": {}
            }
        for key, logs in item["buyers"].items():
            if key not in result[address]["liquidatedWallet"]:
                result[address]["liquidatedWallet"][key] = {}
            for timestamp, liquidation in logs.items():
                liquidation["chainId"] = chain_id
                result[address]["liquidatedWallet"][key][timestamp] = liquidation
        

In [16]:
update_data = []
for address, value in result.items():
    tmp = {
        "_id": address,
        "address": address,
        "liquidationLogs": value
    }
    update_data.append(tmp)
local_klg.update_multichain_wallets(update_data)

# Fix deposit, borrow, balance change logs

In [132]:
m = []

for item in local_connection["knowledge_graph"]["multichain_wallets"].find({}):
    data = item.get("lendings")
    if not data:
        print(item.get("_id"))
        continue
    deposit_logs = {}
    borrow_logs = {}
    deposit_pool_logs = {}
    borrow_pool_logs = {}
    timestamps = []
    for pool, value in data.items():
        chain_id = pool.split("_")[0]
        if chain_id not in ["0x1", "0x38", "0xa4b1", "0x89"]:
            continue
        keys = list(value["depositChangeLogs"].keys())
        keys.sort(reverse=1)
        deposit_pool_logs[pool] = {i: value["depositChangeLogs"][i] for i in keys}
        timestamps += keys
        keys = list(value["borrowChangeLogs"].keys())
        keys.sort(reverse=1)
        borrow_pool_logs[pool] = {i: value["borrowChangeLogs"][i] for i in keys}
        timestamps += keys
    timestamps = list(set(timestamps))

        
    for timestamp in timestamps:
        deposit_logs[timestamp] = 0
        borrow_logs[timestamp] = 0
        for pool, value in deposit_pool_logs.items():
            for key in value:
                if int(key) <= int(timestamp):
                    deposit_logs[timestamp] += value[key]
                    break
        for pool, value in borrow_pool_logs.items():
            for key in value:
                if int(key) <= int(timestamp):
                    borrow_logs[timestamp] += value[key]
                    break
        
    deposit_logs_keys = list(deposit_logs.keys())
    deposit_logs_keys.sort()
    borrow_logs_keys = list(borrow_logs.keys())
    borrow_logs_keys.sort()
    m.append({
        "_id": item.get("_id"),
        "address": item.get("address"),
        "depositChangeLogs": {i: deposit_logs[i] for i in deposit_logs_keys},
        "borrowChangeLogs": {i: borrow_logs[i] for i in borrow_logs_keys}
    })
                        
                    

In [133]:
local_klg.update_docs("multichain_wallets", m)

In [6]:
cursor = atlas_klg.get_docs(collection="multichain_wallets", filter_={})
tokens_list = []
logs = {}
for wallet in cursor:
    tokens = wallet.get("tokenChangeLogs").keys()
    tokens = list(set(list(tokens)))
    tokens_list += [token for token in tokens if token not in tokens_list]
    logs[wallet.get("address")] = []
    for key, value in wallet.get("tokenChangeLogs").items():
        logs[wallet.get("address")] += list(value.keys())
    logs[wallet.get("address")] = list(set(logs[wallet.get("address")]))
print(f"{len(tokens_list)} tokens")
token_cursor = klg.get_docs(collection="smart_contracts", filter_={"_id":{"$in": tokens_list}})
token_prices = {}
for token in token_cursor:
    token_prices[token.get("address")] = {
        "price": token.get("price", 0),
        "priceChangeLogs": token.get("priceChangeLogs", {})
    }

1436 tokens


In [16]:
import json
import time
cursor = atlas_klg.get_docs(collection="multichain_wallets", filter_={})
update_data = []
for wallet in cursor:
    
    timestamps = logs.get(wallet["_id"])
    timestamps.sort()
    wallet["balanceChangeLogs"] = {timestamp: 0 for timestamp in timestamps}
    for token, value in wallet.get("tokenChangeLogs").items():
        for timestamp, amount in value.items():
            if "valueInUSD" not in amount:
                try:
                    amount["valueInUSD"] = amount.get("amount", 0) * get_price(token, token_prices, int(timestamp))
                except Exception as e:
                    print(wallet["address"])
                    raise e
    for timestamp in timestamps:
        for token, value in wallet.get("tokenChangeLogs").items():
            if timestamp in value:
                wallet["balanceChangeLogs"][timestamp] += value.get(timestamp).get("valueInUSD")
            else:
                keys = list(value.keys())
                keys.sort()
                tmp = 0
                min_=time.time()
                for key in keys:
                    if abs(int(key) - int(timestamp)) < min_:
                        tmp = key
                        min_= abs(int(key) - int(timestamp))
                wallet["balanceChangeLogs"][timestamp] += value.get(tmp).get("valueInUSD")
    update_data.append({
        "_id": wallet.get("_id"),
        "address": wallet.get("address"),
        "balanceChangeLogs": wallet.get("balanceChangeLogs"),
        "tokenChangeLogs": wallet.get("tokenChangeLogs")
    })

In [23]:
local_klg.update_docs("knowledge_graph", update_data)